# Phase 3: Preprocessing Notebook

This notebook demonstrates the preprocessing pipeline. We will use the functions defined in `src.preprocessing` to clean the data, engineer features, build a scikit-learn Pipeline, and save the processed data and pipeline for the next phase (model training).

In [1]:
import pandas as pd
import numpy as np
import joblib
import sys
import os

# Add src to path so we can import our modules
sys.path.append(os.path.abspath('..'))
from src.preprocessing import load_and_clean_data, prepare_features, build_preprocessor

import warnings
warnings.filterwarnings('ignore')

## 1. Load and Clean Data

We'll use our modularized `load_and_clean_data` function, which handles missing targets, parses numbers from strings, and caps extreme outliers (as identified in the EDA phase).

In [2]:
df_clean = load_and_clean_data('../data/original/PH_houses_v2.csv')

print(f"Cleaned dataset shape: {df_clean.shape}")
df_clean.head()

Cleaned dataset shape: (1460, 10)


,Description,Location,Price (PHP),Bedrooms,Bath,Floor_area (sqm),Land_area (sqm),Latitude,Longitude,Link
0,2-Bedroom Unit for Sale in Amisa Private Resid...,"Mactan, Lapu-Lapu",15916180.0,2.0,2.0,74.0,NaN,10.305155,124.011241,https://www.lamudi.com.ph/2-bedroom-unit-for-s...
1,1-Bedroom Unit for Sale in Amisa Private Resid...,"Mactan, Lapu-Lapu",8730355.0,1.0,1.0,41.0,NaN,10.305155,124.011241,https://www.lamudi.com.ph/1-bedroom-unit-for-s...
2,Offices at The Galleon - Prime Unit (11G) for ...,"San Antonio, Pasig",27448000.0,NaN,NaN,82.0,NaN,14.588716,121.060009,https://www.lamudi.com.ph/offices-at-the-galle...
3,1-Bedroom Unit (821) for Sale in Maple at Verd...,"Ugong, Pasig",16011000.0,1.0,NaN,58.0,NaN,14.588882,121.079016,https://www.lamudi.com.ph/1-bedroom-unit-821-f...
4,Alice 2 Storey Townhouse For Sale in Lancaster...,"Tapia, General Trias",1935600.0,3.0,NaN,40.0,40.0,14.355070,120.876425,https://www.lamudi.com.ph/alice-2-storey-townh...


## 2. Feature and Target Preparation

Next, we isolate our features (`X`) and our target (`y`). We're using the Log1p transformation on `Price` to normalize its distribution.

In [3]:
X, y = prepare_features(df_clean)

print("Features (X):")
print(X.head())

print("\nTarget (y) - Log1p of Price:")
print(y.head())

Features (X):
               Location  Bedrooms  Bath  Floor_area (sqm)  Land_area (sqm)  \
0     Mactan, Lapu-Lapu       2.0   2.0              74.0              NaN   
1     Mactan, Lapu-Lapu       1.0   1.0              41.0              NaN   
2    San Antonio, Pasig       NaN   NaN              82.0              NaN   
3          Ugong, Pasig       1.0   NaN              58.0              NaN   
4  Tapia, General Trias       3.0   NaN              40.0             40.0   

    Latitude   Longitude  
0  10.305155  124.011241  
1  10.305155  124.011241  
2  14.588716  121.060009  
3  14.588882  121.079016  
4  14.355070  120.876425  

Target (y) - Log1p of Price:
0    16.582847
1    15.982317
2    17.127804
3    16.588787
4    14.475928
Name: Price (PHP), dtype: float64


## 3. Build and Fit Preprocessing Pipeline

Our pipeline will:
1. Impute missing numerical values (Bedrooms, Bath, Areas) with the median.
2. Scale numerical values using StandardScaler.
3. Target-encode the `Location` categorical variable to handle the high cardinality (113 locations) without exploding dimensionality.

In [4]:
preprocessor = build_preprocessor()

# Fit and transform the data
# Note: target encoder requires both X and y for fitting
X_processed = preprocessor.fit_transform(X, y)

print(f"Processed feature matrix shape: {X_processed.shape}")

Processed feature matrix shape: (1460, 7)


## 4. Save Outputs

We'll save the cleaned dataset to `data/processed/` and the fitted preprocessor pipeline to `models/` so they can be easily loaded during model training and deployment.

In [5]:
# Save cleaned dataframe
cleaned_data_path = '../data/processed/cleaned_data.csv'
df_clean.to_csv(cleaned_data_path, index=False)
print(f"Saved cleaned data to {cleaned_data_path}")

# Save the preprocessor pipeline
preprocessor_path = '../models/preprocessor.pkl'
joblib.dump(preprocessor, preprocessor_path)
print(f"Saved preprocessor to {preprocessor_path}")

Saved cleaned data to ../data/processed/cleaned_data.csv
Saved preprocessor to ../models/preprocessor.pkl


**Next Steps:** 
The data is now clean, scaled, and encoded. The next phase will load this preprocessed data, split it into train/test sets, and perform a 3-algorithm model comparison.